# 🏠 Boston, MA Real Estate Market Analysis
**Exploratory Data Analysis | Data Cleaning | Visualization**

---

## Project Overview
This notebook analyzes a dataset of ~1,500 residential property listings across **15 Boston neighborhoods** (2021–2024).  
We explore pricing trends, market dynamics, and what drives home values using Zillow-style listing data.

### Key Questions We'll Answer:
1. Which Boston neighborhoods have the highest (and lowest) home prices?
2. What features most strongly correlate with price?
3. How have prices trended year-over-year?
4. How accurate are Zillow's Zestimates compared to list prices?
5. Where are homes selling fastest?

### Dataset Features
| Column | Description |
|--------|-------------|
|  | Unique property ID |
|  | City (Boston, MA) |
|  | Boston neighborhood |
|  | Single Family / Condo / Townhouse / Multi-Family |
|  /  | Room counts |
|  | Interior square footage |
|  | Year of construction |
|  | Number of garage spaces |
|  | Pool indicator (0/1) |
|  | HOA dues (NaN if no HOA) |
|  | Asking price in USD |
|  | Zillow automated valuation |
|  | List price ÷ sqft |
|  | Days active before sale/removal |
|  | Year listed (2021–2024) |

**Neighborhoods covered:** Back Bay · Beacon Hill · South Boston · South End · Charlestown · Jamaica Plain · Fenway · Dorchester · East Boston · Allston/Brighton · West Roxbury · Roslindale · Roxbury · Hyde Park · Mattapan

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 20)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans'})

print('Libraries loaded successfully ✅')

## 2. Load the Data

In [ ]:
df = pd.read_csv('zillow_listings.csv')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Data Cleaning

In [ ]:
# --- Step 1: Check missing values ---
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(missing_df[missing_df['missing_count'] > 0])

In [ ]:
from IPython.display import Image
Image('images/01_missing_values.png')

In [ ]:
# --- Step 2: Handle missing values ---

# year_built: fill with median (reasonable for age-based calculations)
median_year = df['year_built'].median()
df['year_built'] = df['year_built'].fillna(median_year)
df['year_built'] = df['year_built'].astype(int)

# hoa_monthly_fee: NaN means NO HOA — fill with 0
df['hoa_monthly_fee'] = df['hoa_monthly_fee'].fillna(0)

# Derived feature: age of home
df['home_age'] = 2024 - df['year_built']

print('Missing values after cleaning:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\nAll missing values resolved ✅')

In [ ]:
# --- Step 3: Check for outliers in price ---
Q1 = df['list_price'].quantile(0.25)
Q3 = df['list_price'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 3 * IQR
upper = Q3 + 3 * IQR

outliers = df[(df['list_price'] < lower) | (df['list_price'] > upper)]
print(f'Extreme price outliers: {len(outliers)} listings')
print(f'  Price range kept: ${lower:,.0f} – ${upper:,.0f}')
print(f'  Min price: ${df["list_price"].min():,.0f}')
print(f'  Max price: ${df["list_price"].max():,.0f}')

In [ ]:
# --- Step 4: Check for duplicates ---
dups = df.duplicated(subset='listing_id').sum()
print(f'Duplicate listing IDs: {dups}')
print(f'\nClean dataset ready: {df.shape[0]:,} rows ✅')

## 4. Exploratory Data Analysis (EDA)

### 4.1 Price by Neighborhood

In [ ]:
neighborhood_stats = df.groupby('neighborhood')['list_price'].agg(['median','mean','count']).sort_values('median', ascending=False)
neighborhood_stats.columns = ['Median Price', 'Mean Price', 'Listings']
neighborhood_stats['Median Price'] = neighborhood_stats['Median Price'].apply(lambda x: f'${x:,.0f}')
neighborhood_stats['Mean Price'] = neighborhood_stats['Mean Price'].apply(lambda x: f'${x:,.0f}')
neighborhood_stats

In [ ]:
Image('images/02_price_by_neighborhood.png')

In [ ]:
Image('images/03_ppsf_violin.png')

> **💡 Insight:** Back Bay and Beacon Hill command the highest prices in Boston, reflecting their historic prestige and central location. Neighborhoods like Mattapan, Hyde Park, and Roxbury offer significantly more affordability, presenting opportunities for first-time buyers.

### 4.2 Price vs. Home Size

In [ ]:
corr_sqft = df['list_price'].corr(df['sqft'])
corr_beds = df['list_price'].corr(df['bedrooms'])
print(f'Correlation — Price vs Sqft:     {corr_sqft:.3f}')
print(f'Correlation — Price vs Bedrooms: {corr_beds:.3f}')

In [ ]:
Image('images/04_price_vs_sqft.png')

### 4.3 Price by Property Type

In [ ]:
type_stats = df.groupby('property_type').agg(
    count=('list_price','count'),
    avg_price=('list_price','mean'),
    avg_sqft=('sqft','mean')
).sort_values('avg_price', ascending=False)
type_stats['avg_price'] = type_stats['avg_price'].apply(lambda x: f'${x:,.0f}')
type_stats['avg_sqft'] = type_stats['avg_sqft'].apply(lambda x: f'{x:,.0f} sqft')
type_stats

In [ ]:
Image('images/05_price_by_type.png')

### 4.4 Days on Market by Neighborhood

In [ ]:
dom_stats = df.groupby('neighborhood')['days_on_market'].agg(['median','mean']).sort_values('median')
dom_stats.columns = ['Median DOM', 'Mean DOM']
dom_stats = dom_stats.round(1)
print(dom_stats)
print(f'\nFastest market: {dom_stats.index[0]}')
print(f'Slowest market: {dom_stats.index[-1]}')

In [ ]:
Image('images/06_dom_by_neighborhood.png')

### 4.5 Correlation Analysis

In [ ]:
Image('images/07_correlation.png')

> **💡 Insight:** `sqft`, `bedrooms`, and `bathrooms` show the strongest positive correlations with price. `days_on_market` has a slight negative correlation — higher-priced homes may sit longer, or move fast in hot markets.

### 4.6 Year-over-Year Price Trend

In [ ]:
yearly = df.groupby('list_year')['list_price'].agg(['median','count'])
yearly.columns = ['Median Price', 'Listings']
yearly['YoY Change'] = yearly['Median Price'].pct_change().apply(lambda x: f'{x*100:.1f}%' if pd.notna(x) else '—')
yearly['Median Price'] = yearly['Median Price'].apply(lambda x: f'${x:,.0f}')
print(yearly)

In [ ]:
Image('images/08_price_trend.png')

### 4.7 Zestimate Accuracy

In [ ]:
df['zestimate_diff_pct'] = (df['zestimate'] - df['list_price']) / df['list_price'] * 100

print('Zestimate vs. List Price Accuracy')
print(f'  Mean error:    {df["zestimate_diff_pct"].mean():+.2f}%')
print(f'  Median error:  {df["zestimate_diff_pct"].median():+.2f}%')
print(f'  Std dev:       {df["zestimate_diff_pct"].std():.2f}%')
within_5 = (df['zestimate_diff_pct'].abs() <= 5).mean() * 100
print(f'  Within ±5%:    {within_5:.1f}% of listings')

In [ ]:
Image('images/09_zestimate_accuracy.png')

## 5. Summary of Findings

| # | Finding |
|---|---------|
| 1 | **Back Bay & Beacon Hill are the priciest neighborhoods** — highest median price and $/sqft in Boston |
| 2 | **Square footage is the #1 price driver** — strongest positive correlation with list price |
| 3 | **Multi-family homes average the highest prices** due to rental income potential |
| 4 | **South Boston & Charlestown saw strong year-over-year appreciation** (2021–2024) |
| 5 | **Zestimate accuracy is strong** — over 70% of listings priced within ±5% of Zestimate |
| 6 | **Days-on-market varies by neighborhood** — high-demand areas like South End move significantly faster |

---

### Next Steps / Future Work
- 🤖 Build a **price prediction model** using Linear Regression or Random Forest
- 🗺️ Add **geographic mapping** with folium (Boston neighborhood polygons)
- 📅 Incorporate **seasonal trends** with monthly data
- 🏘️ Expand to **Greater Boston suburbs** (Cambridge, Somerville, Brookline)

---
*Dataset: Simulated Zillow-style Boston listings (2021–2024) across 15 neighborhoods*